# Model 12 — generate the Kaggle submission

**Upload this notebook, attach the original competition dataset, select CPU, and Run All.** The final output is `/kaggle/working/submission.csv`.

This notebook embeds the exact trained larger-gallery Model 12 weights and the already-computed full-test SIFT graph. **No retraining, extra model dataset, raw images, SIFT download, GPU or Internet connection is needed.** Attach the competition `train.csv`, `test.csv`, `sample_submission.csv`, `miewid_ids.csv` and `miewid_features.npy`. If Kaggle provides them as a single competition input, attach that input normally.

The trained stack improved paired larger-gallery local validation MAP@5 from 0.878358 to 0.907654. That is a local result, not a leaderboard estimate. Dalvin's model architecture, PCA/LDA, feature extraction, three-stage stacking and ranking rule are preserved. The supplied train photos all serve as the reference gallery at inference; test identity labels are never used.

Boosters are stored in portable LightGBM text format, rather than a Windows/scikit-learn pickle. The precomputed SIFT graph uses only supplied descriptors and embeddings; input fingerprints prevent accidentally applying its image indices to different data. PCA/LDA and all model scores are computed in this notebook.

The notebook checks both the in-memory predictions and the CSV after writing: exact sample headers and row order, every test ID once, five distinct space-separated labels, and valid species-specific identities or `new_individual`.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, base64, io, zipfile, importlib

# Defaults work on Kaggle. Set DATA_ROOT only if auto-discovery finds multiple copies.
INPUT_ROOT = Path(os.environ.get('MODEL12_INPUT_ROOT', '/kaggle/input'))
WORK = Path(os.environ.get('MODEL12_WORK', '/kaggle/working'))
DATA_ROOT = Path(os.environ['MODEL12_DATA_ROOT']) if os.environ.get('MODEL12_DATA_ROOT') else None
WORKERS = 4
OUTPUT = WORK / 'submission.csv'
CODE = WORK / 'model12_code'
ASSETS = WORK / 'model12_assets'
CODE.mkdir(parents=True, exist_ok=True)
ASSETS.mkdir(parents=True, exist_ok=True)


In [ ]:
SOURCES = {'dalvin_core.py': '"""Dalvin Model 3d functions; raw evidence can use a frozen nuisance projection."""\nimport os\nimport re\nimport cv2\nimport lightgbm as lgb\nimport numpy as np\nimport pandas as pd\nfrom multiprocessing import Pool\nfrom scipy import sparse\nfrom scipy.sparse.csgraph import connected_components\nfrom sklearn.decomposition import PCA\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA\nK=15\nNEIGHBOURS=150\nMATCH_RATIO=.85\nRANSAC_PX=15.\nSIFT_PULL=.05\nVIEWS=[\'left\',\'right\',\'front\',\'back\',\'frontleft\',\'frontright\',\'backleft\',\'backright\',\'up\',\'down\']\nFLANK=np.array([0,1,2,2,0,1,0,1,2,2])\nREGIMES=[None,(.95,111),(.99,89),(.9,67)]\nPARAMS=dict(objective=\'binary\',learning_rate=.05,num_leaves=31,min_child_samples=30,subsample=.8,subsample_freq=1,colsample_bytree=.8,n_estimators=400,verbose=-1,random_state=42,n_jobs=4)\nINDEX_COLS=re.compile(r\'^(nb|nw|sb|sw)\\d$\')\n\n\n\n\n\n\n\ndef unit(a):\n    return a / np.linalg.norm(a, axis=1, keepdims=True)\n\n\ndef embed(gallery, labels, queries):\n    pca = PCA(min(384, gallery.shape[1], len(gallery) + len(queries) - 1), random_state=0).fit(np.r_[gallery, queries])      # unsupervised, sees the queries too\n    lda = LDA(solver="eigen", shrinkage=0.3).fit(pca.transform(gallery), labels)\n    project = lambda a: unit(lda.transform(pca.transform(a)))\n    return project(gallery), project(queries)\n\n\ndef per_identity(sim, inv, n):\n    """Per query: max and mean-of-top-2 similarity to each identity\'s gallery images."""\n    best, top2 = np.empty((sim.shape[0], n)), np.empty((sim.shape[0], n))\n    for j in range(n):\n        s = sim[:, inv == j]\n        best[:, j], top2[:, j] = s.max(1), np.sort(s, 1)[:, -2:].mean(1)\n    return best, top2\n\n\ndef species_features(gi, qi, sp):\n    """Query frame and (query, candidate) frame for one species; gi/qi are global image indices."""\n    Rg, Rq, yg, vg, vq = X[gi], X[qi], Y[gi], VIEW[gi], VIEW[qi]\n    Lg, Lq = embed(Rg, yg, Rq)\n    Rg, Rq = NX[gi], NX[qi]  # Frozen nuisance representation for raw evidence only.\n    ids, inv = np.unique(yg, return_inverse=True)\n    n, nq = len(ids), len(qi)\n    sim_l, sim_r = Lq @ Lg.T, Rq @ Rg.T\n    lda1, lda2 = per_identity(sim_l, inv, n)\n    raw1, raw2 = per_identity(sim_r, inv, n)\n    centroid = unit(np.stack([Lg[inv == j].mean(0) for j in range(n)]))\n    cen = Lq @ centroid.T\n    inl_g, good_g = INL[qi][:, gi].toarray().astype(np.float32), GOOD[qi][:, gi].toarray().astype(np.float32)\n    sift1, sift2 = per_identity(inl_g, inv, n)\n    good1, _ = per_identity(good_g, inv, n)\n    support = (inl_g >= 7).astype(np.float32) @ np.eye(n, dtype=np.float32)[inv]     # gallery images confirmed per identity\n    same_view = np.full((nq, n), np.nan)\n    for j in range(n):\n        for v in np.unique(vq):\n            cols = np.where((inv == j) & (vg == v))[0]\n            if len(cols):\n                same_view[vq == v, j] = sim_l[np.ix_(vq == v, cols)].max(1)\n    # viewpoint coverage: flanks of one animal look different, so unseen sides cannot be matched\n    by_view = np.stack([np.bincount(inv[vg == v], minlength=n) for v in range(len(VIEWS))], 1)\n    by_flank = np.stack([by_view[:, FLANK == f].sum(1) for f in range(3)], 1)\n    seen = vq >= 0\n    n_view = np.where(seen[:, None], by_view[:, vq.clip(0)].T, np.nan)\n    n_flank = np.where(seen[:, None], by_flank[:, FLANK[vq.clip(0)]].T, np.nan)\n    # transductive: neighbours among the queries themselves\n    qq_r, qq_l, qq_s = Rq @ Rq.T, Lq @ Lq.T, INL[qi][:, qi].toarray().astype(np.float32)\n    np.fill_diagonal(qq_r, -1), np.fill_diagonal(qq_l, -1)\n    nb, sb = np.argsort(-qq_l, 1)[:, :5], np.argsort(-qq_s, 1)[:, :5]\n    nbs, sbs = np.take_along_axis(qq_l, nb, 1), np.take_along_axis(qq_s, sb, 1)\n    t_l = np.take_along_axis(lda1, np.argsort(-lda1, 1)[:, :3], 1)\n    t_r = -np.sort(-raw1, 1)[:, :3]\n    t_s = -np.sort(-sift1, 1)[:, :3]\n    g_top = -np.sort(-sim_l, 1)\n    q = pd.DataFrame({\n        "sp": int(sp == "hyena"), "view": vq, "ng": len(gi),\n        "l1": t_l[:, 0], "l2": t_l[:, 1], "l3": t_l[:, 2], "r1": t_r[:, 0], "r2": t_r[:, 1], "r3": t_r[:, 2],\n        "lm12": t_l[:, 0] - t_l[:, 1], "rm12": t_r[:, 0] - t_r[:, 1],\n        "gd5": g_top[:, :5].mean(1), "gd20": g_top[:, :20].mean(1),\n        "qq1": nbs[:, 0], "qq3": nbs[:, :3].mean(1), "qq5": nbs.mean(1), "qqr1": qq_r.max(1),\n        "qn6": (qq_r > .6).sum(1), "qn7": (qq_r > .7).sum(1), "qn8": (qq_r > .8).sum(1),\n        "qq_minus_g": nbs[:, 0] - t_l[:, 0], "nb_g1": t_l[nb[:, 0], 0],\n        "s1": t_s[:, 0], "s2": t_s[:, 1], "s3": t_s[:, 2], "sg": good_g.max(1),\n        "qs1": sbs[:, 0], "qs3": sbs[:, :3].mean(1), "qsn10": (qq_s >= 10).sum(1), "qsn20": (qq_s >= 20).sum(1),\n        "qs_minus_s": sbs[:, 0] - t_s[:, 0], "nbs_s1": t_s[sb[:, 0], 0],\n    })\n    for k in range(5):\n        q[f"nb{k}"], q[f"nw{k}"], q[f"sb{k}"], q[f"sw{k}"] = nb[:, k], nbs[:, k], sb[:, k], sbs[:, k]\n    # candidates: best identities by embedding similarity, lightly boosted by SIFT evidence\n    cand = np.argsort(-(lda1 + SIFT_PULL * np.tanh(sift1 / 15)), 1)[:, :K]\n    r, c = np.repeat(np.arange(nq), K), cand.ravel()\n    nb_ev = np.max(np.minimum(nbs[:, :, None], np.stack([lda1[nb[:, k]] for k in range(5)], 1)), 1)\n    sb_ev = np.max(np.minimum(sbs[:, :, None], np.stack([sift1[sb[:, k]] for k in range(5)], 1)), 1)\n    rank_of = lambda m: (-m).argsort(1).argsort(1)[r, c]\n    cd = pd.DataFrame({\n        "q": r, "c": c, "rank": np.tile(np.arange(K), nq), "ident": ids[c],\n        "mxr": raw1[r, c], "m2r": raw2[r, c], "mxl": lda1[r, c], "m2l": lda2[r, c], "cl": cen[r, c],\n        "gapl": lda1[r, c] - t_l[r, 0], "gapr": raw1[r, c] - t_r[r, 0], "same": same_view[r, c],\n        "cnt": np.bincount(inv)[c], "nbev": nb_ev[r, c], "sbev": sb_ev[r, c],\n        "rk_r": rank_of(raw1), "rk_c": rank_of(cen), "rk_s": rank_of(sift1),\n        "n_view": n_view[r, c], "n_flank": n_flank[r, c], "support": support[r, c], "sx": sift1[r, c], "s2x": sift2[r, c], "gx": good1[r, c], "sgap": sift1[r, c] - t_s[r, 0],\n    })\n    return q, cd\n\n\ndef build_frames(gi, qi):\n    qs, cs, order, off = [], [], [], 0\n    for sp in ("hyena", "leopard"):\n        g, qq = gi[SP[gi] == sp], qi[SP[qi] == sp]\n        q, c = species_features(g, qq, sp)\n        c["q"] += off\n        q[[f"{p}{k}" for p in ("nb", "sb") for k in range(5)]] += off\n        qs.append(q), cs.append(c), order.append(qq)\n        off += len(qq)\n    return dict(q=pd.concat(qs, ignore_index=True), c=pd.concat(cs, ignore_index=True), order=np.concatenate(order))\n\n\ndef duplicate_groups(ix, link):\n    """Group images of one identity that are near-duplicates (same encounter); each image alone if link is None."""\n    if link is None:\n        return np.arange(len(ix))\n    edges = (X[ix] @ X[ix].T > link[0]) | (INL[ix][:, ix].toarray() >= link[1])\n    return connected_components(sparse.csr_matrix(edges))[1]\n\n\ndef columns(s):\n    qf = [c for c in s["q"].columns if c != "unk" and not INDEX_COLS.match(c)]\n    cf = [c for c in s["c"].columns if c not in ("q", "c", "ident", "y")]\n    return qf, cf\n\n\ndef candidate_matrix(s, qf, cf):\n    return pd.concat([s["c"][cf].reset_index(drop=True), s["q"].iloc[s["c"].q.to_numpy()][qf].reset_index(drop=True)], axis=1)\n\n\ndef fit(sims):\n    """new-individual detector, closed-set candidate scorer, and all-query candidate scorer."""\n    qf, cf = columns(sims[0])\n    known = lambda s: s["q"].unk.to_numpy()[s["c"].q.to_numpy()] == 0\n    cx = pd.concat([candidate_matrix(s, qf, cf) for s in sims])\n    cy = np.concatenate([s["c"].y.to_numpy() for s in sims])\n    ck = np.concatenate([known(s) for s in sims])\n    detector = lgb.LGBMClassifier(**{**PARAMS, "n_estimators": 300, "num_leaves": 15}).fit(\n        pd.concat([s["q"][qf] for s in sims]), np.concatenate([s["q"].unk.to_numpy() for s in sims]))\n    return detector, lgb.LGBMClassifier(**PARAMS).fit(cx[ck], cy[ck]), lgb.LGBMClassifier(**PARAMS).fit(cx, cy), (qf, cf)\n\n\ndef predict(models, s):\n    detector, closed, anyq, (qf, cf) = models\n    nq, cx = len(s["q"]), candidate_matrix(s, qf, cf)\n    p_new = detector.predict_proba(s["q"][qf])[:, 1]\n    raw = anyq.predict_proba(cx)[:, 1].reshape(nq, K)\n    within = closed.predict_proba(cx)[:, 1].reshape(nq, K)\n    within /= np.maximum(within.sum(1, keepdims=True), 1e-9)\n    return dict(pn=p_new, raw=raw, pc=within, pb=(p_new + np.clip(1 - raw.sum(1), 0, 1)) / 2)\n\n\ndef rank(s, r):\n    """Top-5 labels: P(new) vs (1 - P(new)) * P(identity | known)."""\n    nq = len(s["q"])\n    labels = np.concatenate([s["c"].ident.to_numpy(dtype=object).reshape(nq, K), np.full((nq, 1), "new_individual", dtype=object)], 1)\n    scores = np.concatenate([(1 - r["pb"])[:, None] * r["pc"], r["pb"][:, None]], 1)\n    return np.take_along_axis(labels, np.argsort(-scores, 1)[:, :5], 1)\n\n\ndef map5(preds, truth):\n    return np.mean([next((1 / (i + 1) for i, p in enumerate(row) if p == t), 0.0) for row, t in zip(preds, truth)])\n\n\ndef augment(s, r):\n    """Add the previous round\'s predictions for each query and for its nearest query neighbours."""\n    q, c, nq = s["q"].copy(), s["c"].copy(), len(s["q"])\n    codes = pd.factorize(c.ident)[0].reshape(nq, K)\n    q["r_pn"], q["r_pb"], q["r_rawsum"], q["r_rawmax"] = r["pn"], r["pb"], r["raw"].sum(1), r["raw"].max(1)\n    c["r_raw"], c["r_pc"] = r["raw"].ravel(), r["pc"].ravel()\n    for kind, weight, scale in (("nb", "nw", 1.0), ("sb", "sw", 30.0)):     # embedding / SIFT neighbours\n        evidence, support, pb, w, top = np.zeros((nq, K)), np.zeros((nq, K)), [], [], []\n        for k in range(5):\n            idx, sim = q[f"{kind}{k}"].to_numpy(), q[f"{weight}{k}"].to_numpy() / scale\n            wk = np.clip(sim - 0.4, 0, None) if kind == "nb" else np.clip(sim, 0, 1)\n            agree = ((codes[idx][:, None, :] == codes[:, :, None]) * r["raw"][idx][:, None, :]).sum(2)\n            evidence = np.maximum(evidence, np.minimum(agree, np.clip(sim, 0, 1)[:, None]))\n            support += wk[:, None] * agree\n            pb.append(r["pb"][idx]), w.append(wk), top.append(r["raw"][idx].max(1))\n        pb, w = np.stack(pb, 1), np.stack(w, 1)\n        q[f"{kind}_pb0"], q[f"{kind}_pbw"] = pb[:, 0], (pb * w).sum(1) / np.maximum(w.sum(1), 1e-6)\n        q[f"{kind}_pbmin"], q[f"{kind}_w"], q[f"{kind}_rm0"] = np.where(w > 0, pb, 1).min(1), w.sum(1), top[0]\n        c[f"r_{kind}ev"], c[f"r_{kind}sum"] = evidence.ravel(), support.ravel()\n    return dict(s, q=q, c=c)\n', 'data_protocol.py': '"""Data loading, role-local mutual SIFT and identity-disjoint episode helpers."""\nimport os\nimport hashlib\nfrom pathlib import Path\nfrom concurrent.futures import ThreadPoolExecutor\nfrom functools import lru_cache\nfrom threading import Lock\nimport numpy as np\nimport pandas as pd\nimport cv2\nfrom scipy import sparse\nimport dalvin_core as b\nNEW=\'new_individual\'\ndef find_sift(root, split):\n    """Kaggle mounts SIFT beside the metadata folder, sometimes as extracted NPZ directories."""\n    name = f"sift_features_{split}.npz"\n    for parent in (root, root.parent, root.parent.parent):\n        candidate = parent / name\n        if candidate.exists():\n            return candidate\n    raise FileNotFoundError(f"Cannot find {name} in {root} or its two parent directories. "\n                            "Attach the full projectwild3 competition input, including both SIFT folders.")\n\n\nclass SiftSource:\n    def __init__(self, path):\n        self.archive = None\n        self.files = {}\n        if path.is_file():\n            self.archive = np.load(path)\n        else:\n            for file in path.rglob("*.npy"):\n                if file.stem in self.files:\n                    raise ValueError(f"Duplicate SIFT key {file.stem} in {path}")\n                self.files[file.stem] = file\n            if not self.files:\n                raise ValueError(f"No extracted .npy SIFT arrays in {path}")\n\n    def __getitem__(self, key):\n        return self.archive[key] if self.archive is not None else np.load(self.files[key], allow_pickle=False)\n\n\ndef load_data(root):\n    train = pd.read_csv(root / "train.csv")\n    test = pd.read_csv(root / "test.csv")\n    ids = pd.read_csv(root / "miewid_ids.csv")\n    all_df = pd.concat([train, test.assign(identity=None)], ignore_index=True)\n    if not all_df.id.is_unique or not ids.id.is_unique:\n        raise ValueError("Duplicate image IDs")\n    for col in ("species", "viewpoint", "identity"):\n        if col not in all_df:\n            raise ValueError(f"Missing column: {col}")\n    if train.identity.isna().any() or not set(all_df.species) <= {"hyena", "leopard"}:\n        raise ValueError("Invalid identity/species metadata")\n    raw = np.load(root / "miewid_features.npy", mmap_mode="r")\n    if len(raw) != len(ids) or set(ids.id) != set(all_df.id):\n        raise ValueError("Feature ID mapping does not match metadata")\n    ix = pd.Index(ids.id).get_indexer(all_df.id)\n    X = np.asarray(raw[ix], dtype=np.float32)\n    if not np.isfinite(X).all() or np.any(np.linalg.norm(X, axis=1) == 0):\n        raise ValueError("Features contain nonfinite values or zero vectors")\n    b.X = b.unit(X)\n    b.IDS = all_df.id.to_numpy()\n    b.Y = all_df.identity.to_numpy(dtype=object)\n    b.SP = all_df.species.to_numpy()\n    b.VIEW = all_df.viewpoint.map({v: i for i, v in enumerate(b.VIEWS)}).fillna(-1).to_numpy(int)\n    return train, test, all_df\n\n\nclass SiftCache:\n    """Role-local neighbour selection; thread workers share lazily loaded descriptors."""\n    def __init__(self, root, out, workers, neighbours):\n        paths = [find_sift(root, s) for s in ("train", "test")]\n        print("SIFT sources:", *paths, sep="\\n", flush=True)\n        self.z = [SiftSource(path) for path in paths]\n        self.read_lock = Lock()\n        self.workers, self.neighbours = workers, neighbours\n        fingerprint = hashlib.sha256(b.IDS.tobytes() if b.IDS.dtype != object else "\\n".join(b.IDS).encode())\n        fingerprint.update(np.ascontiguousarray(b.X).tobytes())\n        self.path = out / ("sift_mutual_" + fingerprint.hexdigest()[:16] + ".npz")\n        self.done = {}\n        if self.path.exists():\n            with np.load(self.path) as z:\n                self.done = {tuple(p): tuple(v) for p, v in zip(z["pairs"], z["values"])}\n\n    @lru_cache(maxsize=1200)\n    def descriptors(self, i):\n        iid = b.IDS[i]\n        z = self.z[0 if iid.startswith("tr_") else 1]\n        # NpzFile\'s shared ZIP handle is not safe for concurrent reads.\n        with self.read_lock:\n            d = np.asarray(z[f"{iid}_desc"], dtype=np.float32)\n            xy = np.asarray(z[f"{iid}_xy"], dtype=np.float32)\n        if d.ndim != 2 or xy.shape != (len(d), 2):\n            raise ValueError(f"Malformed SIFT arrays for {iid}")\n        d = np.sqrt(d / np.maximum(d.sum(1, keepdims=True), 1))\n        return d, xy\n\n    def verify(self, pair):\n        a, c = pair\n        da, xa = self.descriptors(a)\n        dc, xc = self.descriptors(c)\n        if min(len(da), len(dc)) < 4:\n            return 0, 0\n        bf = cv2.BFMatcher()\n        back = np.array([m.trainIdx for m in bf.match(dc, da)])\n        matches = bf.knnMatch(da, dc, k=2)\n        good = [m[0] for m in matches if len(m) == 2 and m[0].distance < .85*m[1].distance\n                and back[m[0].trainIdx] == m[0].queryIdx]\n        if len(good) < 4:\n            return len(good), 0\n        # Local RNG seed per task. OpenCV RNG state is thread-local.\n        cv2.setRNGSeed(int((int(a)*100003 + int(c)) % (2**31-1)))\n        _, mask = cv2.estimateAffinePartial2D(\n            np.float32([xa[m.queryIdx] for m in good]),\n            np.float32([xc[m.trainIdx] for m in good]),\n            method=cv2.RANSAC, ransacReprojThreshold=15.)\n        return len(good), int(mask.sum()) if mask is not None else 0\n\n    def activate(self, indices):\n        wanted = set()\n        for sp in ("hyena", "leopard"):\n            ix = indices[b.SP[indices] == sp]\n            # Chunked similarities avoid allocating the full role x role matrix.\n            for start in range(0, len(ix), 256):\n                q = ix[start:start+256]\n                s = b.X[q] @ b.X[ix].T\n                s[np.arange(len(q)), np.arange(start, start+len(q))] = -np.inf\n                k = min(self.neighbours, len(ix)-1)\n                nb = np.argsort(-s, axis=1)[:, :k]\n                for a, row in zip(q, nb):\n                    wanted.update((min(int(a), int(c)), max(int(a), int(c))) for c in ix[row])\n        missing = sorted(wanted - self.done.keys())\n        print(f"SIFT: {len(wanted)} role-local pairs; {len(missing)} uncached", flush=True)\n        with ThreadPoolExecutor(max_workers=self.workers) as pool:\n            for j, (pair, value) in enumerate(zip(missing, pool.map(self.verify, missing)), 1):\n                self.done[pair] = value\n                if j % 5000 == 0:\n                    self.save()\n                    print(f"  verified {j}/{len(missing)}", flush=True)\n        self.save()\n        # Only expose requested pairs, even when cache contains other roles/test.\n        pp = np.array(sorted(wanted), dtype=int).reshape(-1, 2)\n        vv = np.array([self.done[tuple(p)] for p in pp]).reshape(-1, 2)\n        n = len(b.X)\n        def matrix(v):\n            a, c = pp.T\n            return sparse.coo_matrix((np.r_[v, v], (np.r_[a, c], np.r_[c, a])), shape=(n, n)).tocsr()\n        b.GOOD, b.INL = matrix(vv[:, 0]), matrix(vv[:, 1])\n\n    def save(self):\n        """Persist completed verifications atomically so interrupted runs can resume."""\n        pairs = sorted(self.done)\n        tmp = self.path.with_suffix(".tmp.npz")\n        np.savez_compressed(tmp, pairs=np.array(pairs, dtype=np.int64).reshape(-1, 2),\n                            values=np.array([self.done[p] for p in pairs]).reshape(-1, 2))\n        os.replace(tmp, self.path)\n\n\ndef allocate_roles(train, rounds, seed):\n    rng = np.random.default_rng(seed)\n    roles = {f"stage{i+1}": [] for i in range(rounds)} | {"calibration": [], "validation": []}\n    for sp in ("hyena", "leopard"):\n        names = np.array(sorted(train.loc[train.species == sp, "identity"].unique()))\n        rng.shuffle(names)\n        n = max(1, int(round(.2*len(names))))\n        roles["validation"].extend(names[:n])\n        roles["calibration"].extend(names[n:2*n])\n        for i, block in enumerate(np.array_split(names[2*n:], rounds)):\n            roles[f"stage{i+1}"].extend(block)\n    sets = [set(v) for v in roles.values()]\n    assert all(not a & c for i, a in enumerate(sets) for c in sets[i+1:])\n    return {k: np.flatnonzero(train.identity.isin(v).to_numpy()) for k, v in roles.items()}\n\n\ndef split_episode(indices, seed, cap=32, link=(.95, 100)):\n    """Unknown identities absent from gallery; known-query duplicate groups kept whole."""\n    rng = np.random.default_rng(seed)\n    gallery, known, unknown = [], [], []\n    for sp in ("hyena", "leopard"):\n        ix = indices[b.SP[indices] == sp]\n        names = np.unique(b.Y[ix])\n        rng.shuffle(names)\n        nu = max(1, int(round(rng.uniform(.2, .35)*len(names))))\n        unseen = set(names[:nu])\n        for name in names:\n            rows = ix[b.Y[ix] == name].copy()\n            rng.shuffle(rows)\n            if name in unseen:\n                unknown.extend(rows[:cap])\n                continue\n            held = []\n            groups = b.duplicate_groups(rows, link)\n            target = min(max(1, int(round(.35*len(rows)))), cap)\n            for c in rng.permutation(np.unique(groups)):\n                members = rows[groups == c].tolist()\n                if len(held) < target and len(held)+len(members) < len(rows):\n                    held.extend(members)\n            # Never truncate a duplicate group after selecting it.\n            known.extend(held)\n            gallery.extend(i for i in rows if i not in set(held))\n    gi, qi = np.array(gallery, dtype=int), np.array(known+unknown, dtype=int)\n    if not len(known) or not len(unknown):\n        raise ValueError("Episode lacks known/unknown queries; use a larger role or different seed")\n    for sp in ("hyena", "leopard"):\n        if len(np.unique(b.Y[gi[b.SP[gi] == sp]])) < b.K or np.sum(b.SP[qi] == sp) < 6:\n            raise ValueError("Too few identities/queries per species; reduce --rounds or --candidates")\n    assert not set(b.Y[unknown]) & set(b.Y[gi])\n    assert set(b.Y[known]) <= set(b.Y[gi])\n    return gi, qi, np.where(np.isin(qi, unknown), NEW, b.Y[qi])\n\n\ndef frame(gi, qi, truth=None):\n    s = b.build_frames(gi, qi)\n    if truth is not None:\n        lookup = dict(zip(qi, truth))\n        s["truth"] = np.array([lookup[i] for i in s["order"]])\n        s["q"]["unk"] = (s["truth"] == NEW).astype(int)\n        s["c"]["y"] = (s["c"].ident.to_numpy() == s["truth"][s["c"].q.to_numpy()]).astype(int)\n    return s\n\n\ndef through_stack(s, models):\n    for i, m in enumerate(models):\n        r = b.predict(m, s)\n        if i < len(models)-1:\n            s = b.augment(s, r)\n    return s, r\n', 'submission.py': '"""Portable Model 12 inference and strict competition CSV validation."""\nimport hashlib,json,time,os\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport lightgbm as lgb\nfrom scipy import sparse\nimport dalvin_core as b\nimport data_protocol as p\n\nNEW=\'new_individual\'\n\ndef sha(path):\n    digest=hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda:stream.read(4*1024*1024),b\'\'):digest.update(chunk)\n    return digest.hexdigest()\n\ndef canonical(frame,columns):\n    rows=frame[columns].fillna(\'\').astype(str).values.tolist()\n    return hashlib.sha256(json.dumps(rows,separators=(\',\',\':\'),ensure_ascii=True).encode()).hexdigest()\n\ndef fingerprint(root):\n    root=Path(root)\n    return dict(train=canonical(pd.read_csv(root/\'train.csv\'),[\'id\',\'identity\',\'species\',\'viewpoint\']),\n        test=canonical(pd.read_csv(root/\'test.csv\'),[\'id\',\'species\',\'viewpoint\']),\n        feature_ids=canonical(pd.read_csv(root/\'miewid_ids.csv\'),[\'id\']),\n        feature_array=sha(root/\'miewid_features.npy\'))\n\nclass NativeHead:\n    def __init__(self,path,workers):\n        self.model=lgb.Booster(model_file=str(path));self.workers=workers\n    def predict_proba(self,frame):\n        prob=np.asarray(self.model.predict(frame,num_threads=self.workers,validate_features=True))\n        if prob.ndim!=1 or not np.isfinite(prob).all():raise ValueError(\'Invalid binary model output\')\n        return np.column_stack([1-prob,prob])\n\ndef load_models(assets,workers=4):\n    assets=Path(assets);schema=json.loads((assets/\'model_schema.json\').read_text())\n    models=[]\n    for stage in schema[\'stages\']:\n        heads=[NativeHead(assets/name,workers) for name in stage[\'heads\']]\n        models.append(tuple(heads)+((stage[\'query_features\'],stage[\'candidate_features\']),))\n    if len(models)!=3:raise ValueError(\'Expected the complete three-stage Model 12\')\n    return models\n\ndef prepare(root,assets,sample_path=None):\n    root=Path(root);assets=Path(assets)\n    expected=json.loads((assets/\'input_fingerprint.json\').read_text())\n    actual=fingerprint(root)\n    if actual!=expected:\n        changed=[k for k in expected if actual.get(k)!=expected[k]]\n        raise ValueError(f\'Competition input differs from the trained-model/SIFT bundle: {changed}. Attach the original competition data.\')\n    train,test,all_df=p.load_data(root);b.NX=b.X\n    sample=pd.read_csv(sample_path or root/\'sample_submission.csv\',dtype=str,keep_default_na=False)\n    if list(sample.columns)!=[\'id\',\'predictions\']:raise ValueError(\'Expected sample_submission columns: id,predictions\')\n    if not sample.id.is_unique or set(sample.id)!=set(test.id):raise ValueError(\'Sample IDs do not exactly match test IDs\')\n    with np.load(assets/\'full_test_sift_graph.npz\',allow_pickle=False) as z:\n        pairs=z[\'pairs\'];good=z[\'good\'];inliers=z[\'inliers\'];n=int(z[\'image_count\'])\n    if n!=len(all_df) or pairs.ndim!=2 or pairs.shape[1]!=2:raise ValueError(\'Incorrect SIFT graph dimensions\')\n    if not (len(pairs)==len(good)==len(inliers)):raise ValueError(\'Incomplete SIFT graph\')\n    if not ((pairs[:,0]>=0)&(pairs[:,0]<pairs[:,1])&(pairs[:,1]<n)).all():raise ValueError(\'Invalid SIFT graph indices\')\n    if not (pairs[:,1]>=len(train)).all():raise ValueError(\'Graph contains unrelated training-only edges\')\n    a,c=pairs.T\n    def matrix(v):return sparse.coo_matrix((np.r_[v,v],(np.r_[a,c],np.r_[c,a])),shape=(n,n)).tocsr()\n    b.GOOD=matrix(good);b.INL=matrix(inliers)\n    gi=np.arange(len(train));qi=np.arange(len(train),len(all_df))\n    print(\'Preparing full-gallery features:\',len(train),\'reference photos,\',len(test),\'test photos\',flush=True)\n    frame=p.frame(gi,qi)\n    return dict(train=train,test=test,sample=sample,frame=frame)\n\ndef validate_submission(submission,sample,test,train):\n    if list(submission.columns)!=list(sample.columns) or list(sample.columns)!=[\'id\',\'predictions\']:\n        raise ValueError(\'Submission headers must be exactly id,predictions, with no index column\')\n    if len(submission)!=len(sample) or len(submission)!=len(test):raise ValueError(\'Incorrect submission row count\')\n    if submission.isna().any().any():raise ValueError(\'Submission contains missing values\')\n    if not submission.id.is_unique:raise ValueError(\'Duplicate submission IDs\')\n    if submission.id.tolist()!=sample.id.tolist():raise ValueError(\'Submission IDs must follow the exact sample order\')\n    if set(submission.id)!=set(test.id):raise ValueError(\'Missing or unexpected test IDs\')\n    species=test.set_index(\'id\').species\n    catalog=train.groupby(\'species\').identity.apply(set).to_dict()\n    for iid,value in zip(submission.id,submission.predictions):\n        if not isinstance(value,str):raise ValueError(f\'{iid}: predictions must be a string\')\n        labels=value.split()\n        if len(labels)!=5 or len(set(labels))!=5:raise ValueError(f\'{iid}: expected exactly five distinct labels\')\n        if value!=\' \'.join(labels):raise ValueError(f\'{iid}: use single spaces between prediction labels\')\n        allowed=catalog[species.loc[iid]]|{NEW}\n        invalid=set(labels)-allowed\n        if invalid:raise ValueError(f\'{iid}: unknown or wrong-species identity labels: {invalid}\')\n    return dict(format_valid=True,rows=len(submission),columns=list(submission.columns),\n        unique_ids=True,exact_sample_order=True,labels_per_row=5,distinct_labels=True,\n        labels_valid_for_species=True,index_column_absent=True)\n\ndef generate(context,models,output):\n    start=time.monotonic();frame=context[\'frame\']\n    s,r=p.through_stack(frame,models);pred=b.rank(s,r)\n    predictions=pd.DataFrame({\'id\':b.IDS[s[\'order\']],\'predictions\':[\' \'.join(row) for row in pred]})\n    submission=context[\'sample\'][[\'id\']].merge(predictions,on=\'id\',how=\'left\',sort=False,validate=\'one_to_one\')\n    checks=validate_submission(submission,context[\'sample\'],context[\'test\'],context[\'train\'])\n    output=Path(output);output.parent.mkdir(parents=True,exist_ok=True)\n    temporary=output.with_name(output.stem+\'.checked.tmp.csv\')\n    submission.to_csv(temporary,index=False,encoding=\'utf-8\',lineterminator=\'\\n\')\n    reread=pd.read_csv(temporary,dtype=str,keep_default_na=False)\n    validate_submission(reread,context[\'sample\'],context[\'test\'],context[\'train\'])\n    os.replace(temporary,output)\n    checks.update(csv_sha256=sha(output),prediction_seconds=time.monotonic()-start,\n        new_individual_first=int(submission.predictions.str.split().str[0].eq(NEW).sum()))\n    output.with_suffix(\'.validation.json\').write_text(json.dumps(checks,indent=2))\n    print(\'Saved and re-read validated submission:\',output,flush=True)\n    print(json.dumps(checks,indent=2),flush=True)\n    return submission,checks\n'}
SOURCE_HASHES = {'dalvin_core.py': 'bfeb3384aefce320d2980e68f85b00ea5f109b2c2d4cee7e70d79f05e17a3f1f', 'data_protocol.py': 'fba94b7e73b87e60ee83f3bcb4a5cc01b48082533d3c7d6f420e4e305da4ed72', 'submission.py': '7c8cf67c95e31c00fd355dd37e01f1e32e2c4b010346d5c227e3247eec201462'}
PAYLOAD_SHA256 = 'd36b720d1a27754d0b7e3196bfa2cab1bec918a6af17ad0a5b9fcd50a5e8052d'
PAYLOAD_B64 = '''
...
'''

for name, text in SOURCES.items():
    if hashlib.sha256(text.encode()).hexdigest() != SOURCE_HASHES[name]:
        raise RuntimeError('Embedded source checksum mismatch: ' + name)
    (CODE/name).write_text(text, encoding='utf-8')
archive_bytes = base64.b64decode(PAYLOAD_B64)
if hashlib.sha256(archive_bytes).hexdigest() != PAYLOAD_SHA256:
    raise RuntimeError('Embedded trained-model bundle is corrupted')
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    for info in archive.infolist():
        target = (ASSETS/info.filename).resolve()
        if not target.is_relative_to(ASSETS.resolve()):
            raise RuntimeError('Invalid embedded archive path')
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(info))
bundle = json.loads((ASSETS/'bundle_manifest.json').read_text())
for name, expected_hash in bundle['files'].items():
    if hashlib.sha256((ASSETS/name).read_bytes()).hexdigest() != expected_hash:
        raise RuntimeError('Extracted asset checksum mismatch: ' + name)
sys.path.insert(0, str(CODE))
print('Verified embedded Model 12:', bundle['test_images'], 'test photos,', bundle['sift_graph_pairs'], 'SIFT pairs')


In [ ]:
# Standard Kaggle CPU images provide these packages; nothing is downloaded.
for name in ['numpy', 'pandas', 'scipy', 'sklearn', 'lightgbm', 'cv2', 'threadpoolctl']:
    module = importlib.import_module(name)
    print(name, getattr(module, '__version__', 'available'))
print('Versions used for the validated training run:')
print(json.dumps(json.loads((ASSETS/'training_provenance.json').read_text())['packages'], indent=2))

if DATA_ROOT is None:
    options = sorted({p.parent for p in INPUT_ROOT.rglob('miewid_features.npy')
                      if all((p.parent/name).is_file() for name in ['train.csv', 'test.csv', 'miewid_ids.csv'])})
    if len(options) != 1:
        raise RuntimeError(f'Set DATA_ROOT in the settings cell; found these competition folders: {options}')
    DATA_ROOT = options[0]
required = ['train.csv', 'test.csv', 'miewid_ids.csv', 'miewid_features.npy']
for name in required:
    if not (DATA_ROOT/name).is_file(): raise FileNotFoundError(DATA_ROOT/name)
SAMPLE = DATA_ROOT/'sample_submission.csv'
if not SAMPLE.is_file():
    options = list(INPUT_ROOT.rglob('sample_submission.csv'))
    if len(options) != 1: raise RuntimeError('Attach the competition sample_submission.csv alongside the data')
    SAMPLE = options[0]
print('Competition data:', DATA_ROOT)
print('Sample submission:', SAMPLE)


In [ ]:
import submission as inference
import cv2
from threadpoolctl import threadpool_limits

cv2.setNumThreads(1)
with threadpool_limits(limits=WORKERS):
    context = inference.prepare(DATA_ROOT, ASSETS, SAMPLE)
    models = inference.load_models(ASSETS, WORKERS)
    submission, validation = inference.generate(context, models, OUTPUT)


In [ ]:
from IPython.display import display, FileLink
display(submission.head(10))
print('Submission format checks:', json.dumps(validation, indent=2))
display(FileLink(str(OUTPUT)))
display(FileLink(str(OUTPUT.with_suffix('.validation.json'))))
print('Submit submission.csv from the notebook Output files. This notebook does not upload it automatically.')


## Submit the output

After Run All finishes successfully, save the notebook version to preserve its outputs. Use `submission.csv` in the notebook Output files for the competition submission. Do not submit the notebook or the validation JSON as the prediction file.

The CSV has exactly `id,predictions` as its header. Each predictions field contains five unique identity labels separated by single spaces. `new_individual` is allowed at any ranked position. IDs follow `sample_submission.csv` exactly; the pandas index is excluded.

If the input-fingerprint check fails, verify that the attached data is the original competition input rather than modified feature arrays, reordered feature IDs or a different competition version. The embedded graph belongs to that exact input. Numerical PCA/LDA results can vary slightly with the notebook runtime's library versions; the model weights and matching graph stay fixed.